# AdaBoost Model (With Automatic Tuning)



In [ ]:
import os
import sys
import importlib
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

sys.path.append(os.path.abspath('..'))


import src.model_utils as model_utils
importlib.reload(model_utils)

evaluate_classifier = model_utils.evaluate_classifier
get_confusion_matrix_df = model_utils.get_confusion_matrix_df
load_encoded_splits = model_utils.load_encoded_splits
plot_metric_scorecards = model_utils.plot_metric_scorecards
plot_roc_curve = model_utils.plot_roc_curve
run_random_search = model_utils.run_random_search

In [ ]:
DATA_DIR = '../data/'
X_train, y_train, X_val, y_val, X_test, y_test = load_encoded_splits(DATA_DIR)

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}, y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

In [ ]:
SEED = 42

base_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    random_state=SEED,
)

param_distributions = {
    'n_estimators': [50, 100, 150, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
    'estimator__max_depth': [1, 2, 3],
    'estimator__min_samples_split': [2, 5, 10],
    'estimator__min_samples_leaf': [1, 2, 4],
}

search = run_random_search(
    estimator=base_ada,
    param_distributions=param_distributions,
    X_train=X_train,
    y_train=y_train,
    n_iter=12,
    cv=3,
    scoring='f1',
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)

best_ada = search.best_estimator_
print('Best params:', search.best_params_)
print(f'Best CV f1: {search.best_score_:.4f}')

In [ ]:
val_results = evaluate_classifier(best_ada, X_val, y_val, 'validation')
test_results = evaluate_classifier(best_ada, X_test, y_test, 'test')
plot_metric_scorecards(val_results, test_results)
plot_roc_curve(best_ada, X_val, y_val, 'validation')
plot_roc_curve(best_ada, X_test, y_test, 'test')
val_cm = get_confusion_matrix_df(best_ada, X_val, y_val, 'validation', plot=True)
test_cm = get_confusion_matrix_df(best_ada, X_test, y_test, 'test', plot=True)

results_df = pd.DataFrame([val_results, test_results])
display(results_df.round(4))

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_ada.feature_importances_,
}).sort_values('importance', ascending=False)

display(importance_df.head(15))